# Reachy kinematics test (forward + inverse)
Frame: origin in the torso, **x forward, y to Reachy's left, z up**, units = metres. Joint angles are degrees.

Run top to bottom. Nothing moves until section 4, and section 4 only moves if `DRY_RUN = False`.

In [ ]:
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from findIP import get_reachy_ip
import numpy as np, time

np.set_printoptions(precision=3, suppress=True)
reachy = ReachySDK(host=get_reachy_ip())

ARM_NAME = 'l_arm'
ARM = reachy.l_arm
CHAIN = list(ARM.kinematics_chain.values())   # the 7 arm joints (gripper excluded)
print([j.name for j in CHAIN])

In [ ]:
def current_q():
    return [j.present_position for j in CHAIN]

def describe(T, label=''):
    x, y, z = T[:3, 3]
    print(f'{label} position (m): x={x:.3f}  y={y:.3f}  z={z:.3f}')
    print('rotation:\n', T[:3, :3])

def pose_error(A, B):
    """Position error (mm) and rotation error (deg) between two 4x4 poses."""
    pos_mm = np.linalg.norm(A[:3, 3] - B[:3, 3]) * 1000
    R = A[:3, :3].T @ B[:3, :3]
    ang = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))
    return pos_mm, ang

## 1. Forward kinematics: where is the hand right now?
The arm goes limp. Move it around by hand and re-run the second cell to watch the numbers change. Hand forward = x goes up, hand up = z goes up.

In [ ]:
reachy.turn_off(ARM_NAME)

In [ ]:
q = current_q()
print('joints (deg):', np.round(q, 1))
T_now = ARM.forward_kinematics()      # uses present positions
describe(T_now, 'hand')

## 2. FK then IK round trip (no motion)
The arm has 7 joints, so IK can return a *different* set of angles for the same pose. So we compare the **pose** the IK answer gives, not the angles.

In [ ]:
q_ik = ARM.inverse_kinematics(T_now, q0=q)
T_check = ARM.forward_kinematics(joints_position=q_ik)
print('original q:', np.round(q, 1))
print('IK q      :', np.round(q_ik, 1))
pos_mm, rot_deg = pose_error(T_now, T_check)
print(f'pose error: {pos_mm:.1f} mm, {rot_deg:.2f} deg  (expect ~0)')

## 3. Plan a small Cartesian move (no motion yet)
Same hand orientation, position shifted by `OFFSET`. Start small. Put the arm by hand somewhere in front of Reachy with the elbow bent, away from the table and torso, before running this.

In [ ]:
OFFSET = np.array([0.0, 0.0, 0.05])   # metres: +5 cm straight up
MAX_STEP_M = 0.10                      # refuse anything larger than 10 cm
MAX_JOINT_JUMP = 45                    # refuse plans that swing any joint > 45 deg

assert np.linalg.norm(OFFSET) <= MAX_STEP_M, 'offset too big for a test'

q_start = current_q()
T_start = ARM.forward_kinematics(joints_position=q_start)
T_goal = T_start.copy()
T_goal[:3, 3] += OFFSET

q_goal = ARM.inverse_kinematics(T_goal, q0=q_start)   # q0 = stay close to current posture
jump = np.abs(np.array(q_goal) - np.array(q_start))
describe(T_goal, 'goal')
for j, a, b, d in zip(CHAIN, q_start, q_goal, jump):
    print(f'{j.name:18s} {a:7.1f} -> {b:7.1f}   ({d:5.1f})')
assert jump.max() < MAX_JOINT_JUMP, 'IK picked a big swing; try a different starting posture'
print('plan OK')

## 4. Execute and measure accuracy
`turn_on` makes the arm stiff where it is, then it moves to the goal and comes back.

In [ ]:
DRY_RUN = True   # set to False when the plan above looks sane

def move_to(q_target, duration=2.0):
    goto({j: a for j, a in zip(CHAIN, q_target)}, duration=duration,
         interpolation_mode=InterpolationMode.MINIMUM_JERK)

if not DRY_RUN:
    reachy.turn_on(ARM_NAME)
    move_to(q_goal, 2.0)
    time.sleep(0.5)
    T_reached = ARM.forward_kinematics()
    pos_mm, rot_deg = pose_error(T_goal, T_reached)
    print(f'reached goal within {pos_mm:.1f} mm, {rot_deg:.2f} deg')
    print('actual displacement (m):', T_reached[:3, 3] - T_start[:3, 3])
    move_to(q_start, 2.0)

## 5. Go to an absolute point (the thing grasping needs)
Keeps the current hand orientation and goes to `TARGET_XYZ`. Read real values for a spot on your table from section 1 first, then put them here.

In [ ]:
TARGET_XYZ = np.array([0.30, 0.20, -0.25])   # example only: replace with your own measured point

T_target = ARM.forward_kinematics()  # borrow the current orientation
T_target[:3, 3] = TARGET_XYZ
try:
    q_target = ARM.inverse_kinematics(T_target, q0=current_q())
    print('reachable. joints:', np.round(q_target, 1))
    print(f'IK accuracy: {pose_error(T_target, ARM.forward_kinematics(joints_position=q_target))[0]:.1f} mm')
except ValueError as e:
    print('IK failed (probably out of reach):', e)

# when happy:
# reachy.turn_on(ARM_NAME); move_to(q_target, 3.0)

In [ ]:
# Done: let the arm down gently (same arm we used)
reachy.turn_off_smoothly(ARM_NAME)